# 01 - Online Retail II: Data Preparation

**Dataset**: Online Retail II (UK-based online retailer, giftware, mostly wholesale)
**Source**: UCI Machine Learning Repository
**Transactions**: 1,067,371 lines | **Period**: Dec. 2009 - Dec. 2011 | **Variables**: 9
**Currency**: GBP (£)

---
## Business Context & Objectives

This notebook loads the **Online Retail II** dataset and establishes the foundational data cleaning decisions that all downstream analyses (customer segmentation in Notebook 02, retention/churn in 03, repurchase prediction in 04) depend on.

### Key Objectives
1. **Raw observation:** explore the raw data (schema, types, missingness, volume) without applying any filter yet.
2. **Data integrity & edge cases:** identify and handle retail-specific anomalies (invoice cancellations, non-product stock codes, zero/negative prices).
3. **Traceability:** quantify the impact of each cleaning decision in both rows removed and revenue affected.
4. **Decision logging:** document every assumption so downstream notebooks inherit an auditable, not a silent, set of choices.

## Research Questions

1. **Data quality**: what does a transaction line actually represent, and which lines are unusable for customer-level analysis?
2. **Scope**: once cleaned, what customer base and what share of revenue remains for segmentation, retention and prediction (notebooks 02-04)?

---
### Performance note
The raw `.xlsx` (~45 MB) is read once and cached locally as Parquet. Re-running the notebook then takes seconds instead of minutes.

---
## Key Variables

| Type | Variables |
|---|---|
| **Transaction** | `Invoice`, `InvoiceDate`, `Quantity`, `Price` |
| **Product** | `StockCode`, `Description` |
| **Customer** | `Customer ID`, `Country` |
| **Derived** | `IsCancellation`, `IsNonProduct`, `LineRevenue` |

---
## Plan

0. Setup & Loading
1. General Overview
2. Data Quality & Retail Edge Cases
3. Data Cleaning & Transformation Pipeline
4. Summary & Decision Log


# 0. Setup & Loading

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# -----------------------------------------------------------------------------
# Paths configuration
# -----------------------------------------------------------------------------
RAW = Path("../data/raw/online_retail_II.xlsx")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = PROC / "raw_concat.parquet"

# -----------------------------------------------------------------------------
# Data loading & caching logic
# -----------------------------------------------------------------------------
if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
    source_info = f"Loaded dataset from Parquet cache (`{PARQUET_PATH.name}`)"
else:
    sheets = pd.read_excel(RAW, sheet_name=None)
    df = pd.concat(
        [sheet.assign(SourceSheet=name) for name, sheet in sheets.items()],
        ignore_index=True,
    )

    # Clean column headers
    df.columns = df.columns.str.strip()

    # Cast object columns to string to fix mixed types before Parquet export:
    # - 'Invoice': contains integers and cancellation codes ('C489449')
    # - 'StockCode': contains numeric codes and non-product codes (POST, DOT, M...)
    # - 'Description': contains a few numeric values among free text
    # The 'string' dtype preserves missing values as <NA> instead of literal "nan".
    text_cols = df.select_dtypes(include="object").columns
    df[text_cols] = df[text_cols].apply(lambda s: s.astype("string").str.strip())

    # Cache dataset
    df.to_parquet(PARQUET_PATH, index=False)
    source_info = f"Parsed raw Excel sheets and cached to `{PARQUET_PATH.name}`"

# -----------------------------------------------------------------------------
# Execution feedback
# -----------------------------------------------------------------------------
display(Markdown(f"### Data Loading Completed\n* **Source:** {source_info}"))


### Data Loading Completed
* **Source:** Loaded dataset from Parquet cache (`raw_concat.parquet`)

# 1. General Overview

First, a general look at the raw dataset's shape and schema before any processing.

## 1.1 Dataset Summary & Schema

In [2]:
sheets_list = ", ".join(df["SourceSheet"].unique())
min_date = df["InvoiceDate"].min()
max_date = df["InvoiceDate"].max()

summary = f"""
### Dataset Summary
* **Dimensions:** `{df.shape[0]:,}` rows x `{df.shape[1]}` columns
* **Source sheets:** {sheets_list}
* **Date range:** from `{min_date}` to `{max_date}`
"""
display(Markdown(summary))
display(df.head())



### Dataset Summary
* **Dimensions:** `1,067,371` rows x `9` columns
* **Source sheets:** Year 2009-2010, Year 2010-2011
* **Date range:** from `2009-12-01 07:45:00` to `2011-12-09 12:50:00`


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010


## 1.2 Proactive Type Checking and Identifier Inspection

Before casting types or performing calculations, we check that identifiers such as `Invoice` or `StockCode` are homogeneous. A proactive check tells us whether these columns contain non-numeric characters, which would indicate a naming convention or distinct event types encoded in the same field.

In [3]:
display(Markdown("### Column Types"))

dtypes_df = pd.DataFrame(df.dtypes, columns=["Data Type"]).reset_index()
dtypes_df.columns = ["Column", "Type"]
display(dtypes_df)


### Column Types

,Column,Type
0,Invoice,string[python]
1,StockCode,string[python]
2,Description,string[python]
3,Quantity,int64
4,InvoiceDate,datetime64[ns]
5,Price,float64
6,Customer ID,float64
7,Country,string[python]
8,SourceSheet,string[python]


In [4]:
non_num_invoices = df[df["Invoice"].str.contains(r"[A-Za-z]", na=False)]
non_num_stockcodes = df[df["StockCode"].str.contains(r"[A-Za-z]", na=False)]

display(Markdown(f"* **Invoices with letters:** `{len(non_num_invoices):,}` ({len(non_num_invoices)/len(df):.2%})"))
display(Markdown(f"* **StockCodes with letters:** `{len(non_num_stockcodes):,}` ({len(non_num_stockcodes)/len(df):.2%})"))

display(Markdown("### Non-numeric characters in Invoice"))
display(non_num_invoices.head())

display(Markdown("### Non-numeric characters in StockCode"))
display(non_num_stockcodes.head())


* **Invoices with letters:** `19,500` (1.83%)

* **StockCodes with letters:** `134,986` (12.65%)

### Non-numeric characters in Invoice

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Year 2009-2010
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,Year 2009-2010
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,Year 2009-2010
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,Year 2009-2010
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Year 2009-2010


### Non-numeric characters in StockCode

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
12,489436,48173C,DOOR MAT BLACK FLOCK,10,2009-12-01 09:06:00,5.95,13078.0,United Kingdom,Year 2009-2010
23,489436,35004B,SET OF 3 BLACK FLYING DUCKS,12,2009-12-01 09:06:00,4.65,13078.0,United Kingdom,Year 2009-2010
28,489436,84596F,SMALL MARSHMALLOWS PINK BOWL,8,2009-12-01 09:06:00,1.25,13078.0,United Kingdom,Year 2009-2010


**Observation:** the non-numeric invoices start with the letter `C`. Their quantities are negative. This indicates cancellation transactions (`C` = Cancellation). They are isolated as a distinct signal rather than silently netted against regular sales — the netting decision, if any, is made explicitly in notebook 02, not here.

# 1.3 Missing Values & Uniqueness Audit

In [5]:
# -----------------------------------------------------------------------------
# 1.3 Missing values & revenue impact audit
# -----------------------------------------------------------------------------

# Line-level revenue. fillna(0) is defensive: Quantity and Price have no
# missing values in this dataset, but this keeps the calculation safe if
# that were ever not the case.
line_revenue = (df["Quantity"] * df["Price"]).fillna(0)
total_revenue = line_revenue.sum()

n_missing = df.isna().sum()
pct_missing = (n_missing / len(df) * 100).round(2)

# Vectorized missing-revenue per column: df.isna().T (cols x rows) @ line_revenue (rows x 1)
revenue_missing = df.isna().T.dot(line_revenue)
pct_revenue_missing = (revenue_missing / total_revenue * 100).fillna(0).round(2)

missing_summary = pd.DataFrame({
    "Missing Values": n_missing,
    "Percentage (%)": pct_missing,
    "Revenue Missing (GBP)": revenue_missing,
    "Revenue Missing (%)": pct_revenue_missing,
}).sort_values(by="Missing Values", ascending=False)

display(Markdown("### Missing Values & Financial Impact Breakdown"))
display(
    missing_summary.style.format({
        "Missing Values": "{:,}",
        "Percentage (%)": "{:.2f}%",
        "Revenue Missing (GBP)": "£{:,.2f}",
        "Revenue Missing (%)": "{:.2f}%",
    })
)

display(Markdown(f"""
### Unique Entities
* **Unique customers:** `{df['Customer ID'].nunique():,}`
* **Unique invoices:** `{df['Invoice'].nunique():,}`
* **Unique stock codes:** `{df['StockCode'].nunique():,}`
"""))


C:\Users\jboul\AppData\Local\Temp\ipykernel_23196\651018095.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  pct_revenue_missing = (revenue_missing / total_revenue * 100).fillna(0).round(2)


### Missing Values & Financial Impact Breakdown

,Missing Values,Percentage (%),Revenue Missing (GBP),Revenue Missing (%)
Customer ID,"243,007",22.77%,"£2,638,958.18",13.68%
Description,"4,382",0.41%,£0.00,0.00%
Invoice,0,0.00%,£0.00,0.00%
Quantity,0,0.00%,£0.00,0.00%
StockCode,0,0.00%,£0.00,0.00%
InvoiceDate,0,0.00%,£0.00,0.00%
Price,0,0.00%,£0.00,0.00%
Country,0,0.00%,£0.00,0.00%
SourceSheet,0,0.00%,£0.00,0.00%



### Unique Entities
* **Unique customers:** `5,942`
* **Unique invoices:** `53,628`
* **Unique stock codes:** `5,304`


## First observations

- **1,067,371 transaction lines** across two trading years (Dec. 2009 - Dec. 2011), from a UK-based online retailer selling giftware, mostly to wholesale buyers.
- **`Customer ID` is missing on 22.77% of lines** (guest checkouts or non-attributed sales). These lines are unusable for any customer-level analysis (RFM, retention, churn, repurchase prediction) since there is no entity to attach them to, but remain valid for product- or revenue-level questions. They are excluded when the customer-level table is built (section 3).
- **Missing-ID rows are lower value on average**: 22.77% of rows but only 13.68% of revenue, roughly 0.60x the average line value of identified transactions. Consistent with unregistered one-off purchases in a customer base that is otherwise wholesale-heavy, where large buyers are systematically identified. Excluding these rows removes a disproportionately low-value segment, not a representative slice of revenue.
- **`Description` is missing on 0.41% of lines** — negligible, no action needed.


# 2. Data Quality & Retail Edge Cases

Investigating specific retail anomalies: cancellations, price/quantity edge cases, and non-product stock codes.

## 2.1 Cancellation Analysis (`C` Invoice Prefix)

In [6]:
cancellations = df[df["Invoice"].str.startswith("C", na=False)]
cancellation_revenue = (cancellations["Quantity"] * cancellations["Price"]).sum()

display(Markdown(f"""
### Cancellation Summary (`C` prefix)
* **Total cancellation rows:** `{len(cancellations):,}` ({len(cancellations)/len(df):.2%})
* **Total negative quantity:** `{cancellations['Quantity'].sum():,}`
* **Net financial impact:** `£{cancellation_revenue:,.2f}`
"""))



### Cancellation Summary (`C` prefix)
* **Total cancellation rows:** `19,494` (1.83%)
* **Total negative quantity:** `-490,992`
* **Net financial impact:** `£-1,526,667.86`


## 2.2 Price and Quantity Anomalies

In [7]:
# Check 1: negative quantities without a 'C' prefix (returns not coded as cancellations?)
neg_qty_no_c = df[(df["Quantity"] < 0) & (~df["Invoice"].str.startswith("C", na=False))]

# Check 2: cancellations ('C') with positive quantities (already found one instance, see below)
pos_qty_with_c = df[(df["Quantity"] > 0) & (df["Invoice"].str.startswith("C", na=False))]

# Check 3: zero or negative prices
zero_or_neg_price = df[df["Price"] <= 0]

display(Markdown(f"""
### Data Integrity Anomalies
* **Negative quantity without 'C':** `{len(neg_qty_no_c):,}` rows
* **Cancellation ('C') with positive quantity:** `{len(pos_qty_with_c):,}` rows
* **Zero or negative price:** `{len(zero_or_neg_price):,}` rows
"""))



### Data Integrity Anomalies
* **Negative quantity without 'C':** `3,457` rows
* **Cancellation ('C') with positive quantity:** `1` rows
* **Zero or negative price:** `6,207` rows


**Interpretation — pending.** These three counts have not been interpreted yet: no decision should be made before seeing the actual figures. In particular:

- If `neg_qty_no_c` is non-trivial, these may be returns processed without the cancellation convention, and deserve their own decision rather than being silently caught by a blanket `Quantity > 0` filter later (which would also strip out the cancellations we deliberately keep, see 2.1 above).
- `pos_qty_with_c` — one such row was already found and explained in section 2.1 of the original exploration (`C496350`, `StockCode = "M"`, no `Customer ID` attached, and thus excluded regardless in section 3). If this count is exactly 1, no further action is needed; if higher, each case should be checked individually as before, not assumed.
- `zero_or_neg_price` rows carry no revenue and cannot inform monetary value; whether to exclude them depends on their volume and on whether `Quantity` is otherwise meaningful on those rows (e.g. free promotional items).

**Action:** run the cell above, report the three counts and inspect a sample of each (`.head()`), before deciding what — if anything — is excluded on this basis. No filter is applied on price or quantity sign in the cleaning pipeline (section 3) until this is resolved.

## 2.3 Non-Product Codes Analysis

`StockCode` may mix genuine products with codes representing administrative or non-merchandise entries (postage, manual adjustments, fees). This is checked directly below rather than assumed, since these codes have no meaning for product-level or customer-behaviour analysis.

In [8]:
non_product_codes = df.loc[
    df["StockCode"].str.match(r"^[A-Za-z]+$", na=False),
    "StockCode"
].value_counts()

display(Markdown("### Non-Product Codes Overview"))
display(Markdown(f"*Found **`{len(non_product_codes)}`** unique alphabetic stock codes.*"))

npc_df = non_product_codes.reset_index()
npc_df.columns = ["StockCode", "Occurrence Count"]
display(npc_df)


### Non-Product Codes Overview

*Found **`16`** unique alphabetic stock codes.*

,StockCode,Occurrence Count
0,POST,2122
1,DOT,1446
2,M,1421
3,D,177
4,S,104
5,ADJUST,67
6,AMAZONFEE,43
7,DCGSSGIRL,25
8,DCGSSBOY,23
9,PADS,19


The alpha-only filter over-catches: `DCGSSGIRL`, `DCGSSBOY`, `DCGSLGIRL`, `DCGSLBOY` (a childrenswear line) and `PADS` (cushion pads) are genuine products, coded with letters only, and are **kept**. `GIFT` is also kept as a sellable item, not an accounting entry.

Confirmed non-product / administrative codes, excluded from the analytical base: `POST`, `DOT` (postage), `M`/`m` (manual entries, merged case-insensitive), `D` (discount), `S` (samples), `ADJUST` (accounting adjustment), `AMAZONFEE` (platform fee), `CRUK` (charity donation). Only codes actually observed above with non-trivial volume are included here — no code is added on the assumption that it "usually" appears in this dataset.

In [9]:
# Confirmed non-product / administrative codes (verified against the value_counts above)
NON_PRODUCT_CODES = {
    "POST": "Postage",
    "DOT": "Dotcom postage / charge",
    "M": "Manual entry",
    "D": "Discount",
    "S": "Samples",
    "ADJUST": "Accounting adjustment",
    "AMAZONFEE": "Amazon platform fee",
    "CRUK": "Cancer Research UK donation",
}

df["StockCodeUpper"] = df["StockCode"].str.upper()
df["IsNonProduct"] = df["StockCodeUpper"].isin(NON_PRODUCT_CODES.keys())

non_product_df = df[df["IsNonProduct"]]

non_product_summary = (
    non_product_df.groupby("StockCodeUpper")
    .apply(
        lambda g: pd.Series({
            "Description": NON_PRODUCT_CODES.get(g.name, "Other"),
            "Total Rows": len(g),
            "Total Quantity": g["Quantity"].sum(),
            "Total Revenue (GBP)": (g["Quantity"] * g["Price"]).sum(),
        }),
        include_groups=False,
    )
    .reset_index()
)

display(Markdown("### Non-Product Codes Overview"))
display(non_product_summary.style.format({
    "Total Rows": "{:,}",
    "Total Quantity": "{:,}",
    "Total Revenue (GBP)": "£{:,.2f}",
}))


### Non-Product Codes Overview

,StockCodeUpper,Description,Total Rows,Total Quantity,Total Revenue (GBP)
0,ADJUST,Accounting adjustment,67,5,"£6,835.24"
1,AMAZONFEE,Amazon platform fee,43,-35,"£-260,763.58"
2,CRUK,Cancer Research UK donation,16,-16,"£-7,933.43"
3,D,Discount,177,"-2,872","£-13,484.54"
4,DOT,Dotcom postage / charge,"1,446","2,938","£322,647.47"
5,M,Manual entry,"1,426","4,612","£-82,781.27"
6,POST,Postage,"2,122","10,108","£112,341.00"
7,S,Samples,104,-98,"£-6,065.80"


**Non-product lines represent 0.51% of rows but +£70,795.09 in signed revenue** — small in volume, and their net effect happens to be positive, but this masks large offsetting amounts: `DOT` alone contributes +£322,647 (postage charges, dotcom operations) while `AMAZONFEE` contributes -£260,763 across only 43 lines (average -£6,064 per line — large platform-fee debits, not ordinary transactions). `M`, `D`, `S`, and `CRUK` are net negative, as expected for manual adjustments, discounts, samples, and a charity donation.

These lines carry no product or customer-behaviour information and are excluded from the analytical base used in notebooks 02-04. They are kept in the raw table (flagged via `IsNonProduct`, not deleted) so the exclusion remains auditable.

# 3. Data Cleaning & Transformation Pipeline

Building the customer-level analytical base used by notebooks 02-04. Two exclusions are applied together:

1. **Non-product lines** (`IsNonProduct`) excluded.
2. **Rows without `Customer ID`** excluded.

**Cancellations are kept, not excluded here.** Whether to net them into a customer's monetary value or treat cancellation behaviour as a separate signal is analysis-specific, and is decided explicitly in notebook 02 — not baked silently into this shared base. A prior version of this pipeline removed cancellations at this stage; that was a mistake, since it contradicts the decision stated in section 2.1 and would make cancellation behaviour invisible to every downstream notebook.

The price/quantity anomalies from section 2.2 are **not yet applied** — pending the actual counts (see the note in that section).

## 3.1 Filtering & Feature Engineering

In [10]:
# -----------------------------------------------------------------------------
# 3.1 Data cleaning pipeline & feature engineering
# -----------------------------------------------------------------------------

initial_rows = len(df)
total_raw_revenue = (df["Quantity"] * df["Price"]).sum()

# Ensure derived flags exist even if this cell is re-run independently
if "StockCodeUpper" not in df.columns:
    df["StockCodeUpper"] = df["StockCode"].str.upper()
if "IsNonProduct" not in df.columns:
    df["IsNonProduct"] = df["StockCodeUpper"].isin(NON_PRODUCT_CODES.keys())
df["IsCancellation"] = df["Invoice"].str.startswith("C", na=False)

# Step 1: exclude non-product lines
step1 = df.loc[~df["IsNonProduct"]]
rows_after_non_product = len(step1)

# Step 2: exclude rows without a Customer ID
# Cancellations are intentionally NOT filtered out here — they are kept,
# flagged via IsCancellation, so notebook 02 can decide explicitly how to
# treat them (net into Monetary value, or use as a standalone signal).
df_clean = step1.loc[step1["Customer ID"].notna()].copy()
rows_final = len(df_clean)

df_clean["Customer ID"] = df_clean["Customer ID"].astype("int64").astype("string")
df_clean["LineRevenue"] = df_clean["Quantity"] * df_clean["Price"]

# -----------------------------------------------------------------------------
# Summary metrics
# -----------------------------------------------------------------------------
pct_clean = (rows_final / initial_rows) * 100
n_unique_cust = df_clean["Customer ID"].nunique()
revenue_retained = df_clean["LineRevenue"].sum()
pct_revenue_retained = (revenue_retained / total_raw_revenue) * 100
n_cancellations_kept = df_clean["IsCancellation"].sum()

summary_clean = f"""
### Cleaned Dataset Ready
* **Analytical base:** `{rows_final:,}` rows (`{pct_clean:.2f}%` of raw data)
* **Unique customers:** `{n_unique_cust:,}`
* **Revenue retained:** `£{revenue_retained:,.2f}` (`{pct_revenue_retained:.2f}%` of raw total revenue)
* **Cancellation rows kept (flagged):** `{n_cancellations_kept:,}`
"""
display(Markdown(summary_clean))



### Cleaned Dataset Ready
* **Analytical base:** `820,963` rows (`76.91%` of raw data)
* **Unique customers:** `5,882`
* **Revenue retained:** `£16,728,575.72` (`86.73%` of raw total revenue)
* **Cancellation rows kept (flagged):** `17,950`


## 3.2 Clean Dataset Export

In [11]:
out_path = PROC / "clean_transactions.parquet"
df_clean.to_parquet(out_path, index=False)
display(Markdown(f"Saved successfully to: `{out_path}`"))


Saved successfully to: `..\data\processed\clean_transactions.parquet`

# 4. Summary & Decision Log

## 4.1 Impact Quantification

In [12]:
decision_log = pd.DataFrame([
    {
        "Step": "Raw data",
        "Condition": "Initial dataset",
        "Rows Remaining": initial_rows,
        "Rows Removed": 0,
        "% Retained": "100.0%",
    },
    {
        "Step": "1. Non-product codes",
        "Condition": "Remove POST, DOT, M, D, S, ADJUST, AMAZONFEE, CRUK",
        "Rows Remaining": rows_after_non_product,
        "Rows Removed": initial_rows - rows_after_non_product,
        "% Retained": f"{rows_after_non_product/initial_rows:.1%}",
    },
    {
        "Step": "2. Missing Customer ID",
        "Condition": "Remove rows without a Customer ID",
        "Rows Remaining": rows_final,
        "Rows Removed": rows_after_non_product - rows_final,
        "% Retained": f"{rows_final/initial_rows:.1%}",
    },
])

display(Markdown("### Cleaning Decision Log & Impact Summary"))
display(decision_log)


### Cleaning Decision Log & Impact Summary

,Step,Condition,Rows Remaining,Rows Removed,% Retained
0,Raw data,Initial dataset,1067371,0,100.0%
1,1. Non-product codes,"Remove POST, DOT, M, D, S, ADJUST, AMAZONFEE, ...",1061970,5401,99.5%
2,2. Missing Customer ID,Remove rows without a Customer ID,820963,241007,76.9%


## Decision log summary

| # | Decision | Rows affected | Revenue affected |
|---|----------|---------------|-------------------|
| 1 | Non-product codes excluded | 5,401 (0.51%) | +£70,795.09 |
| 2 | Missing Customer ID excluded | 243,007 (22.77%) | +£2,638,958.18 (13.68%) |
| 3 | Cancellations flagged, **kept** (not netted) | 19,494 (1.83%) | -£1,526,667.86 (-7.9%) — retained in the base |
| pending | Price/quantity anomalies (2.2) | not yet quantified | not yet quantified |

The resulting analytical base (`clean_transactions.parquet`) drops only non-product lines and unidentified customers — the two categories with no customer-level meaning — and keeps cancellations as a flagged signal, exactly as decided in section 2.1. This table is the single source for notebooks 02-04, pending resolution of the price/quantity anomaly check above.

## Summary

The analytical base retains 76.91% of raw rows but 86.73% of revenue, across 5,882 uniquely identified customers spanning Dec. 2009 - Dec. 2011, with cancellations preserved and flagged rather than removed. This confirms the earlier observation: excluded rows (non-product entries, unidentified customers) are disproportionately low-value. This table (`clean_transactions.parquet`) is the input to notebook 02, once the pending price/quantity check above is resolved.

In [13]:
display(neg_qty_no_c.head(10))
display(zero_or_neg_price.head(10))
print(zero_or_neg_price["Price"].value_counts().head())

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,Year 2009-2010
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,Year 2009-2010
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,Year 2009-2010
470,489521,21646,<NA>,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,Year 2009-2010
3114,489655,20683,<NA>,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,Year 2009-2010
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom,Year 2009-2010
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom,Year 2009-2010
4296,489806,18010,<NA>,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom,Year 2009-2010
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN,United Kingdom,Year 2009-2010
4566,489821,85049G,<NA>,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom,Year 2009-2010


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,Year 2009-2010
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,Year 2009-2010
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,Year 2009-2010
470,489521,21646,<NA>,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,Year 2009-2010
3114,489655,20683,<NA>,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,Year 2009-2010
3161,489659,21350,<NA>,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom,Year 2009-2010
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom,Year 2009-2010
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom,Year 2009-2010
3731,489781,84292,<NA>,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom,Year 2009-2010
4296,489806,18010,<NA>,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom,Year 2009-2010


Price
 0.00        6202
-11062.06       2
-53594.36       1
-44031.79       1
-38925.87       1
Name: count, dtype: int64


In [14]:
display(df[df["StockCode"].str.upper() == "B"][["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]])

,Invoice,StockCode,Description,Quantity,Price,Customer ID
179403,A506401,B,Adjust bad debt,1,-53594.36,NaN
276274,A516228,B,Adjust bad debt,1,-44031.79,NaN
403472,A528059,B,Adjust bad debt,1,-38925.87,NaN
825443,A563185,B,Adjust bad debt,1,11062.06,NaN
825444,A563186,B,Adjust bad debt,1,-11062.06,NaN
825445,A563187,B,Adjust bad debt,1,-11062.06,NaN


In [15]:
display(neg_qty_no_c.head(10))
display(df[(df["Price"] == 0) & (df["StockCodeUpper"] != "B")].head(10))

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,Year 2009-2010
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,Year 2009-2010
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,Year 2009-2010
470,489521,21646,<NA>,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,Year 2009-2010
3114,489655,20683,<NA>,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,Year 2009-2010
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom,Year 2009-2010
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom,Year 2009-2010
4296,489806,18010,<NA>,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom,Year 2009-2010
4538,489820,21133,invcd as 84879?,-720,2009-12-02 13:23:00,0.0,NaN,United Kingdom,Year 2009-2010
4566,489821,85049G,<NA>,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom,Year 2009-2010


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,StockCodeUpper,IsNonProduct,IsCancellation
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom,Year 2009-2010,21733,False,False
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom,Year 2009-2010,71477,False,False
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom,Year 2009-2010,85123A,False,False
470,489521,21646,<NA>,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom,Year 2009-2010,21646,False,False
3114,489655,20683,<NA>,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom,Year 2009-2010,20683,False,False
3161,489659,21350,<NA>,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom,Year 2009-2010,21350,False,False
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.0,NaN,United Kingdom,Year 2009-2010,35956,False,False
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.0,NaN,United Kingdom,Year 2009-2010,35605A,False,False
3731,489781,84292,<NA>,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom,Year 2009-2010,84292,False,False
4296,489806,18010,<NA>,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom,Year 2009-2010,18010,False,False


In [16]:
pct_neg_qty_no_cust = neg_qty_no_c["Customer ID"].isna().mean() * 100
pct_zero_price_no_cust = df[(df["Price"] == 0) & (df["StockCodeUpper"] != "B")]["Customer ID"].isna().mean() * 100

print(f"neg_qty_no_c sans Customer ID : {pct_neg_qty_no_cust:.2f}%")
print(f"zero_or_neg_price (hors B) sans Customer ID : {pct_zero_price_no_cust:.2f}%")

neg_qty_no_c sans Customer ID : 100.00%
zero_or_neg_price (hors B) sans Customer ID : 98.86%


In [17]:
zero_price_residual = df[
    (df["Price"] == 0)
    & (df["StockCodeUpper"] != "B")
    & (df["Customer ID"].notna())
]

print(f"Lignes résiduelles : {len(zero_price_residual)}")
display(zero_price_residual.head(15))
print(zero_price_residual["Description"].value_counts().head(10))

Lignes résiduelles : 71


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,StockCodeUpper,IsNonProduct,IsCancellation
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126.0,United Kingdom,Year 2009-2010,22076,False,False
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658.0,United Kingdom,Year 2009-2010,48185,False,False
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.0,17231.0,United Kingdom,Year 2009-2010,M,True,False
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108.0,United Kingdom,Year 2009-2010,22065,False,False
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108.0,United Kingdom,Year 2009-2010,22142,False,False
32916,492079,85042,ANTIQUE LILY FAIRY LIGHTS,8,2009-12-15 13:49:00,0.0,15070.0,United Kingdom,Year 2009-2010,85042,False,False
40101,492760,21143,ANTIQUE GLASS HEART DECORATION,12,2009-12-18 14:22:00,0.0,18071.0,United Kingdom,Year 2009-2010,21143,False,False
47126,493761,79320,FLAMINGO LIGHTS,24,2010-01-06 14:54:00,0.0,14258.0,United Kingdom,Year 2009-2010,79320,False,False
48342,493899,22355,"CHARLOTTE BAG , SUKI DESIGN",10,2010-01-08 10:43:00,0.0,12417.0,Belgium,Year 2009-2010,22355,False,False
57619,494607,21533,RETRO SPOT LARGE MILK JUG,12,2010-01-15 12:43:00,0.0,16858.0,United Kingdom,Year 2009-2010,21533,False,False


Description
Manual                            7
CHRISTMAS PUDDING TRINKET POT     2
This is a test product.           2
REGENCY CAKESTAND 3 TIER          2
ROUND CAKE TIN VINTAGE GREEN      2
CHRISTMAS CRAFT WHITE FAIRY       1
ANTIQUE LILY FAIRY LIGHTS         1
FLAMINGO LIGHTS                   1
ANTIQUE GLASS HEART DECORATION    1
CHARLOTTE BAG , SUKI DESIGN       1
Name: count, dtype: Int64


In [18]:
# Extent réel de TEST001 dans tout le dataset, pas seulement ce sous-ensemble à prix nul
test_rows = df[df["StockCodeUpper"] == "TEST001"]
print(f"Lignes TEST001 (tout le dataset) : {len(test_rows)}")
display(test_rows[["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]])

# Recherche plus large : d'autres descriptions évoquant des données de test
test_like = df[df["Description"].str.contains("test", case=False, na=False)]
print(f"\nLignes dont la description contient 'test' : {len(test_like)}")
display(test_like["Description"].value_counts())


Lignes TEST001 (tout le dataset) : 15


,Invoice,StockCode,Description,Quantity,Price,Customer ID
27994,491725,TEST001,This is a test product.,10,4.5,12346.0
28251,491742,TEST001,This is a test product.,5,4.5,12346.0
28254,491744,TEST001,This is a test product.,5,4.5,12346.0
39398,492718,TEST001,This is a test product.,5,4.5,12346.0
45228,493410,TEST001,This is a test product.,5,4.5,12346.0
45230,493412,TEST001,This is a test product.,5,4.5,12346.0
56117,494450,TEST001,This is a test product.,5,4.5,12346.0
66084,495295,TEST001,This is a test product.,5,4.5,12346.0
89084,497819,TEST001,This is a test product.,5,0.0,14103.0
89180,497843,TEST001,This is a test product.,5,0.0,14827.0



Lignes dont la description contient 'test' : 19


Description
This is a test product.    16
test                        3
Name: count, dtype: Int64

In [19]:
# Est-ce que le client 12346 n'achète que du TEST001, ou aussi de vrais produits ?
cust_12346 = df[df["Customer ID"] == 12346.0]
print(f"Total lignes client 12346 : {len(cust_12346)}")
print(f"Dont TEST001 : {(cust_12346['StockCodeUpper'] == 'TEST001').sum()}")
display(cust_12346[cust_12346["StockCodeUpper"] != "TEST001"].head(10))

# Les 3 lignes "test" (minuscule, sans le 001) — même code produit ou autre chose ?
test_lower = df[df["Description"].str.lower() == "test"]
display(test_lower[["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID"]])

Total lignes client 12346 : 48
Dont TEST001 : 9


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet,StockCodeUpper,IsNonProduct,IsCancellation
39411,492722,TEST002,This is a test product.,1,2009-12-18 10:55:00,1.00,12346.0,United Kingdom,Year 2009-2010,TEST002,False,False
71080,C495800,ADJUST,Adjustment by john on 26/01/2010 17,-1,2010-01-26 17:27:00,103.50,12346.0,United Kingdom,Year 2009-2010,ADJUST,True,True
107800,499763,20682,RED SPOTTY CHILDS UMBRELLA,1,2010-03-02 13:08:00,3.25,12346.0,United Kingdom,Year 2009-2010,20682,False,False
107801,499763,20679,EDWARDIAN PARASOL RED,1,2010-03-02 13:08:00,5.95,12346.0,United Kingdom,Year 2009-2010,20679,False,False
107802,499763,15056N,EDWARDIAN PARASOL NATURAL,1,2010-03-02 13:08:00,5.95,12346.0,United Kingdom,Year 2009-2010,15056N,False,False
107803,499763,15056BL,EDWARDIAN PARASOL BLACK,1,2010-03-02 13:08:00,5.95,12346.0,United Kingdom,Year 2009-2010,15056BL,False,False
107804,499763,15056P,EDWARDIAN PARASOL PINK,1,2010-03-02 13:08:00,5.95,12346.0,United Kingdom,Year 2009-2010,15056P,False,False
253028,513774,21524,DOORMAT SPOTTY HOME SWEET HOME,1,2010-06-28 13:53:00,7.49,12346.0,United Kingdom,Year 2009-2010,21524,False,False
253029,513774,22692,DOORMAT WELCOME TO OUR HOME,1,2010-06-28 13:53:00,7.49,12346.0,United Kingdom,Year 2009-2010,22692,False,False
253030,513774,22660,DOORMAT I LOVE LONDON,1,2010-06-28 13:53:00,7.49,12346.0,United Kingdom,Year 2009-2010,22660,False,False


,Invoice,StockCode,Description,Quantity,Price,Customer ID
857865,566064,22355,test,1,0.0,NaN
864418,566573,22823,test,-22,0.0,NaN
864419,566574,22823,test,22,0.0,NaN


In [20]:
TEST_CODES = {"TEST001", "TEST002"}
NON_PRODUCT_CODES = {
    "POST": "Postage",
    "DOT": "Dotcom postage / charge",
    "M": "Manual entry",
    "D": "Discount",
    "S": "Samples",
    "ADJUST": "Accounting adjustment",
    "AMAZONFEE": "Amazon platform fee",
    "CRUK": "Cancer Research UK donation",
    "B": "Bad debt adjustment",
    "TEST001": "Test product (system test data)",
    "TEST002": "Test product (system test data)",
}

In [21]:
test_variants = df[df["StockCodeUpper"].str.contains("TEST", na=False)]
print(test_variants["StockCodeUpper"].value_counts())

StockCodeUpper
TEST001    15
TEST002     2
Name: count, dtype: Int64
